# **Project Title:** AI-Powered IoT Network Anomaly Detection and Investigation Platform

## Phase 3: Train and Evaluate 5-Fold Stacking

| Item | Details |
|---|---|
| Input | Output `stacking_fixed_split_v1` của notebook processing |
| Base models | Decision Tree, Random Forest, HistGradientBoosting |
| Meta-model | Logistic Regression, học trên xác suất OOF |
| OOF | 5 fold dùng chung; preprocessing fit riêng trong từng fold |
| Selection | Macro-F1 trên validation; tie-break theo candidate ID đã chốt |
| Final evaluation | Best stacking và best base model, freeze trước khi mở test |
| Runtime | **Kaggle**, CPU, toàn bộ train; không tự lấy mẫu giảm dữ liệu |

**Chưa chạy trên Kaggle.** Chạy notebook processing trước, lưu output thành Kaggle Dataset và gắn vào đây.
Mỗi base model được fit 5 lượt OOF và 1 lượt toàn train; thời gian có thể dài trên hàng triệu mẫu.

Giữ train/validation/test có sẵn của Kaggle; chỉ tạo 5 fold trong train. Baseline validation/test đã từng được xem.
Không dùng lại `X_*_raw.npy`, `X_*_scaled.npy` hoặc preprocessor đã fit toàn train để tạo OOF.


## Training Protocol

- Giữ split nguồn từ train.csv / validation.csv / test.csv, không chia lại 80/10/10.
- Processing chỉ khử trùng bên trong train và lọc overlap evaluation theo thứ tự train → validation → test; không chuyển mẫu sang tập khác.

- Dùng đúng fold ID do processing xuất; kiểm tra nhóm, checksum, thứ tự lớp và đủ 8 lớp.
- Trong mỗi fold: fit median + bỏ cột hằng trên 4 fold, transform fold OOF, kiểm tra input thực nhận rồi fit từng base model.
- Các base models đều là cây; dùng thang gốc, không log/scaling. `class_weight="balanced"` được tính từ phần fit tương ứng, không dùng SMOTE/undersampling.
- Ghép 8 xác suất của mỗi base model theo thứ tự lớp cố định. Mỗi dòng train có đúng một dự đoán OOF cho mỗi base model.
- Meta-model chỉ học từ OOF và nhãn train. Fit lại base models/preprocessor trên toàn train để dự đoán validation/test.
- Lưới ứng viên, seed và ngân sách được lưu trước khi train. Chọn bằng validation rồi freeze; không sửa cấu hình theo test.
- Lưu per-class precision/recall/F1, macro-F1, confusion matrices, thời gian và phần cứng.

Tham khảo: [StackingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.StackingClassifier.html),
[tránh data leakage](https://scikit-learn.org/stable/common_pitfalls.html),
[StratifiedGroupKFold](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedGroupKFold.html).
Notebook viết vòng lặp OOF tường minh để dễ kiểm tra, không dùng `cv="prefit"` thay OOF.


# Import Library


In [ ]:
import gc
import hashlib
import importlib.util
import json
import os
import platform
import time
import uuid
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from threadpoolctl import threadpool_info, threadpool_limits


## 1. Freeze Experiment Configuration Before Training


In [ ]:
# Thay cấu hình tại đây trước Restart Session → Run All, không sửa sau khi xem test.
DATA_DIR = None  # Ví dụ: Path("/kaggle/input/datasets/<owner>/<dataset>/<run_id>")
RANDOM_STATE = 42
N_FOLDS = 5
N_JOBS = 2
PREDICT_BATCH_SIZE = 25_000
HASH_BATCH_SIZE = 25_000
GOLDEN_SAMPLE_SIZE = 32

CLASS_NAMES = [
    "Normal", "BruteForce", "DDoS", "DoS",
    "Mirai", "Recon", "Spoofing", "Web-Based",
]
CLASS_IDS = np.arange(len(CLASS_NAMES))

base_templates = {
    "DT": DecisionTreeClassifier(
        max_depth=20, min_samples_leaf=2, class_weight="balanced", random_state=RANDOM_STATE
    ),
    "RF": RandomForestClassifier(
        n_estimators=100, max_depth=20, min_samples_leaf=2,
        max_features="sqrt", max_samples=0.8, class_weight="balanced",
        n_jobs=N_JOBS, random_state=RANDOM_STATE,
    ),
    "HGB": HistGradientBoostingClassifier(
        max_iter=100, learning_rate=0.1, max_leaf_nodes=31,
        min_samples_leaf=20, l2_regularization=1.0,
        class_weight="balanced", early_stopping=False, random_state=RANDOM_STATE,
    ),
}
BASE_NAMES = list(base_templates)
BASE_SUBSETS = [["DT", "RF", "HGB"], ["RF", "HGB"]]
META_CONFIGS = [
    {"name": "LR_C1", "C": 1.0, "class_weight": None},
    {"name": "LR_C1_balanced", "C": 1.0, "class_weight": "balanced"},
    {"name": "LR_C01_balanced", "C": 0.1, "class_weight": "balanced"},
]
META_MAX_ITER = 1000

if not Path("/kaggle/input").is_dir() or not Path("/kaggle/working").is_dir():
    raise RuntimeError("Chỉ chạy huấn luyện trên Kaggle.")

RUN_ID = "stacking_" + time.strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:8]
OUTPUT_DIR = Path("/kaggle/working/stacking_runs") / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
SESSION_START = time.perf_counter()
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

print("=" * 70)
print("STACKING EXPERIMENT CONFIGURATION")
print("=" * 70)
print(f"Run ID         : {RUN_ID}")
print(f"Base models    : {', '.join(BASE_NAMES)}")
print(f"OOF folds      : {N_FOLDS}")
print(f"Stack candidates: {len(BASE_SUBSETS) * len(META_CONFIGS)}")
print(f"CPU threads    : {N_JOBS}")
print("Selection      : validation macro-F1")


### Helper Functions — Artifacts and Input Checks


In [ ]:
def write_json(path, content):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(content, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8"
    )


def sha256_file(path):
    checksum = hashlib.sha256()
    with path.open("rb") as file:
        block = file.read(8 * 1024 * 1024)
        while block:
            checksum.update(block)
            block = file.read(8 * 1024 * 1024)
    return checksum.hexdigest()


def check_classes(targets, description):
    if not np.array_equal(np.unique(targets), CLASS_IDS):
        raise ValueError(f"{description}: cần đủ nhãn 0–7 theo thứ tự cố định.")


def check_probabilities(model, probabilities):
    if not np.array_equal(model.classes_, CLASS_IDS):
        raise ValueError("Thứ tự classes_ không khớp 8 lớp.")
    if probabilities.shape[1] != len(CLASS_NAMES) or not np.isfinite(probabilities).all():
        raise ValueError("Ma trận xác suất không hợp lệ.")
    if np.any(probabilities < 0) or np.any(probabilities > 1):
        raise ValueError("Xác suất ngoài [0, 1].")
    if not np.allclose(probabilities.sum(axis=1), 1.0, atol=1e-5):
        raise ValueError("Tổng xác suất mỗi dòng không bằng 1.")


def feature_hashes(values):
    # Cùng quy tắc float32, -0.0 và NaN với notebook processing.
    result = np.empty(len(values), dtype="V16")
    for start in range(0, len(values), HASH_BATCH_SIZE):
        stop = min(start + HASH_BATCH_SIZE, len(values))
        block = np.array(values[start:stop], dtype="<f4", order="C", copy=True)
        block[block == 0] = 0.0
        block[np.isnan(block)] = np.nan
        hashes = []
        for row in block:
            hashes.append(hashlib.blake2b(row.tobytes(), digest_size=16).digest())
        result[start:stop] = np.asarray(hashes, dtype="V16")
    return result


def check_effective_overlap(fit_values, hold_values, description):
    fit_hashes = np.unique(feature_hashes(fit_values))
    hold_hashes = np.unique(feature_hashes(hold_values))
    overlap_count = len(np.intersect1d(fit_hashes, hold_hashes, assume_unique=True))
    if overlap_count > 0:
        raise ValueError(
            f"{description}: {overlap_count} nhóm trùng sau median/chọn cột. "
            "Dừng để xem lại nhóm/split, không tự bỏ dòng OOF hoặc test."
        )
    return {"description": description, "overlap_groups": 0, "dtype": "float32"}


def make_preprocessor():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
        ("constant_filter", VarianceThreshold(threshold=0.0)),
    ])


def fit_preprocessor(values, description):
    for column_index in range(values.shape[1]):
        if np.isnan(values[:, column_index]).all():
            raise ValueError(f"{description}: cột {feature_names[column_index]} hoàn toàn thiếu.")
    preprocessor = make_preprocessor()
    transformed = preprocessor.fit_transform(values).astype(np.float32, copy=False)
    if not np.isfinite(transformed).all():
        raise ValueError(f"{description}: NaN/Inf sau tiền xử lý.")
    selected = preprocessor.named_steps["constant_filter"].get_support()
    kept_features = np.asarray(feature_names)[selected].tolist()
    record = {"fit_part": description, "kept_features": kept_features, "output_dtype": "float32"}
    return preprocessor, transformed, record


## 2. Locate and Verify the Processing Run


In [ ]:
print("=" * 70)
print("VERIFY STACKING INPUT ARTIFACTS")
print("=" * 70)

if DATA_DIR is None:
    candidate_dirs = []
    for path in Path("/kaggle/input").rglob("preprocessing_config.json"):
        candidate = json.loads(path.read_text())
        status_path = path.parent / "run_status.json"
        if candidate.get("protocol") == "stacking_fixed_split_v1" and status_path.is_file():
            status = json.loads(status_path.read_text())
            if status.get("status") == "complete":
                candidate_dirs.append(path.parent)
    if len(candidate_dirs) != 1:
        raise ValueError(f"Cần chọn DATA_DIR rõ ràng. Các run phù hợp: {candidate_dirs}")
    DATA_DIR = candidate_dirs[0]
else:
    DATA_DIR = Path(DATA_DIR)

data_config = json.loads((DATA_DIR / "preprocessing_config.json").read_text())
data_status = json.loads((DATA_DIR / "run_status.json").read_text())
schema = json.loads((DATA_DIR / "feature_schema.json").read_text())
checksums = json.loads((DATA_DIR / "checksums.json").read_text())
if data_config.get("protocol") != "stacking_fixed_split_v1" or data_status.get("status") != "complete":
    raise ValueError("Input phải là run stacking_fixed_split_v1 đã complete.")
if data_status["run_id"] != data_config["run_id"]:
    raise ValueError("Run ID của config và status không khớp.")
if data_config["class_names"] != CLASS_NAMES or schema["class_names"] != CLASS_NAMES:
    raise ValueError("Thứ tự lớp không khớp.")
if data_config["n_folds"] != N_FOLDS or data_config["preprocessing_fitted"]:
    raise ValueError("Cần 5 fold và input chưa fit preprocessing.")
if schema["input_dtype"] != "float32" or data_config["dtype"] != "float32":
    raise ValueError("Input dtype phải là float32.")
if data_config.get("split_policy", {}).get("assignment") != "source_csv":
    raise ValueError("Cần output giữ đúng split CSV nguồn, không phải bản chia lại.")
if data_config["split_policy"].get("rows_redistributed") is not False:
    raise ValueError("Input không được chuyển mẫu giữa các split.")
feature_names = schema["input_features"]
if feature_names != data_config["features"] or len(set(feature_names)) != len(feature_names):
    raise ValueError("Danh sách feature không khớp hoặc bị lặp.")

required_files = [
    "preprocessing_config.json", "feature_schema.json", "label_mapping.json",
    "split_protocol.json", "fold_id_train.npy"
]
for split in ["train", "val", "test"]:
    required_files.extend([
        f"X_{split}_unprocessed.npy", f"y_{split}.npy",
        f"group_id_{split}.npy", f"sample_metadata_{split}.pkl",
    ])
for file_name in required_files:
    expected_checksum = checksums.get(file_name)
    if not isinstance(expected_checksum, str):
        raise ValueError(f"Thiếu checksum cho {file_name}.")
    if sha256_file(DATA_DIR / file_name) != expected_checksum:
        raise ValueError(f"Checksum không khớp: {file_name}")
print(f"Data run: {data_config['run_id']}")
print(f"Input folder: {DATA_DIR}")
print(f"[OK] Verified {len(required_files)} required files.")

X_train = np.load(DATA_DIR / "X_train_unprocessed.npy", mmap_mode="r", allow_pickle=False)
y_train = np.load(DATA_DIR / "y_train.npy", allow_pickle=False)
X_val = np.load(DATA_DIR / "X_val_unprocessed.npy", mmap_mode="r", allow_pickle=False)
y_val = np.load(DATA_DIR / "y_val.npy", allow_pickle=False)
fold_ids = np.load(DATA_DIR / "fold_id_train.npy", allow_pickle=False)
train_groups = np.load(DATA_DIR / "group_id_train.npy", allow_pickle=False)
val_groups = np.load(DATA_DIR / "group_id_val.npy", allow_pickle=False)
# Chỉ dùng group ID để kiểm tra split; chưa đọc X_test hoặc y_test.
test_groups = np.load(DATA_DIR / "group_id_test.npy", allow_pickle=False)

for split, values, targets in [("train", X_train, y_train), ("val", X_val, y_val)]:
    if values.dtype != np.float32 or values.shape != (len(targets), len(feature_names)):
        raise ValueError(f"{split}: shape/dtype không đúng contract.")
    if list(values.shape) != data_config["artifacts"][split]["shape"]:
        raise ValueError(f"{split}: shape không khớp config.")
    check_classes(targets, split)
    for start in range(0, len(values), PREDICT_BATCH_SIZE):
        if np.isinf(values[start:start + PREDICT_BATCH_SIZE]).any():
            raise ValueError(f"{split}: input chứa Inf.")
if len(fold_ids) != len(y_train) or not np.array_equal(np.unique(fold_ids), np.arange(N_FOLDS)):
    raise ValueError("Fold ID không hợp lệ.")
if len(train_groups) != len(y_train) or len(val_groups) != len(y_val):
    raise ValueError("Group ID lệch hàng với target.")
for left, right in [(train_groups, val_groups), (train_groups, test_groups), (val_groups, test_groups)]:
    if len(np.intersect1d(left, right)) > 0:
        raise ValueError("Nhóm bị trùng giữa các split.")
print(f"Train shape: {X_train.shape}")
print(f"Val shape  : {X_val.shape}")
print("Test values and labels will be loaded only after freezing candidates.")


### Record Protocol, Versions and Hardware


In [ ]:
cpu_name = platform.processor()
if Path("/proc/cpuinfo").is_file():
    for line in Path("/proc/cpuinfo").read_text().splitlines():
        if line.startswith("model name"):
            cpu_name = line.split(":", 1)[1].strip()
            break
hardware = {
    "platform": platform.platform(), "cpu": cpu_name,
    "logical_cpus": os.cpu_count(), "requested_threads": N_JOBS,
    "ram_gib": os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1024**3,
    "accelerator_used": "CPU", "threadpools": threadpool_info(),
}
versions = {
    "python": platform.python_version(), "numpy": np.__version__,
    "pandas": pd.__version__, "sklearn": sklearn.__version__, "joblib": joblib.__version__,
}
base_parameters = {}
for name, template in base_templates.items():
    base_parameters[name] = template.get_params(deep=False)
protocol = {
    "run_id": RUN_ID, "data_run_id": data_config["run_id"],
    "data_checksums_sha256": sha256_file(DATA_DIR / "checksums.json"),
    "n_folds": N_FOLDS, "seed": RANDOM_STATE,
    "source_split_policy": data_config["split_policy"],
    "base_parameters": base_parameters, "base_subsets": BASE_SUBSETS,
    "meta_configs": META_CONFIGS, "meta_max_iter": META_MAX_ITER,
    "preprocessing": "fold-specific train median + constant filter; float32; no log/scaling",
    "resampling": "none", "weights": "balanced inside each base-model fit",
    "selection": "highest validation macro-F1, then ascending candidate_id",
    "test_candidates": "best stacking and best individual base model, frozen before test",
    "previous_validation_test_viewed": True, "versions": versions, "hardware": hardware,
}
write_json(OUTPUT_DIR / "protocol.json", protocol)
requirements = [
    "numpy==" + versions["numpy"], "pandas==" + versions["pandas"],
    "scikit-learn==" + versions["sklearn"], "joblib==" + versions["joblib"],
]
(OUTPUT_DIR / "requirements.txt").write_text("\n".join(requirements) + "\n", encoding="utf-8")
write_json(OUTPUT_DIR / "run_status.json", {"status": "in_progress", "run_id": RUN_ID})
print("=" * 70)
print("PROTOCOL FROZEN BEFORE TRAINING")
print("=" * 70)
print(f"CPU: {cpu_name}")
print(f"RAM: {hardware['ram_gib']:.2f} GiB")
print(f"sklearn: {versions['sklearn']}")
print(f"Output: {OUTPUT_DIR}")


### Helper Functions — Probabilities and Evaluation


In [ ]:
def model_probabilities(model, transformed):
    probabilities = np.empty((len(transformed), len(CLASS_NAMES)), dtype=np.float32)
    with threadpool_limits(limits=N_JOBS):
        for start in range(0, len(transformed), PREDICT_BATCH_SIZE):
            stop = min(start + PREDICT_BATCH_SIZE, len(transformed))
            batch_probabilities = model.predict_proba(transformed[start:stop])
            check_probabilities(model, batch_probabilities)
            probabilities[start:stop] = batch_probabilities
    return probabilities


def meta_features(probabilities_by_model, names):
    blocks = []
    for name in names:
        blocks.append(probabilities_by_model[name])
    return np.concatenate(blocks, axis=1)


def evaluate_predictions(candidate_id, split, targets, probabilities, predict_seconds, fit_seconds):
    predicted = probabilities.argmax(axis=1)
    report = classification_report(
        targets, predicted, labels=CLASS_IDS, target_names=CLASS_NAMES,
        output_dict=True, zero_division=0,
    )
    per_class = []
    for class_name in CLASS_NAMES:
        row = {"Class": class_name}
        row.update(report[class_name])
        per_class.append(row)
    per_class_table = pd.DataFrame(per_class)
    metrics = {
        "candidate_id": candidate_id, "split": split,
        "accuracy": float(accuracy_score(targets, predicted)),
        "macro_f1": float(f1_score(targets, predicted, labels=CLASS_IDS, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(targets, predicted, labels=CLASS_IDS, average="weighted", zero_division=0)),
        "fit_seconds": float(fit_seconds), "predict_seconds": float(predict_seconds),
        "rows": len(targets),
        "predict_ms_per_row": float(predict_seconds / len(targets) * 1000),
    }
    directory = OUTPUT_DIR / split / candidate_id
    directory.mkdir(parents=True, exist_ok=True)
    per_class_table.to_csv(directory / "per_class_metrics.csv", index=False)
    write_json(directory / "metrics.json", metrics)
    write_json(directory / "classification_report.json", report)

    matrix = confusion_matrix(targets, predicted, labels=CLASS_IDS)
    row_totals = matrix.sum(axis=1, keepdims=True)
    normalized = np.divide(matrix, row_totals, out=np.zeros_like(matrix, dtype=float), where=row_totals != 0)
    pd.DataFrame(matrix, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(directory / "confusion_matrix.csv")
    pd.DataFrame(normalized, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(directory / "confusion_matrix_normalized.csv")
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    for ax, values, title in [(axes[0], matrix, "Count"), (axes[1], normalized, "Row-normalized")]:
        image = ax.imshow(values, cmap="Blues")
        for row in range(len(CLASS_NAMES)):
            for col in range(len(CLASS_NAMES)):
                if title == "Count":
                    label = f"{matrix[row, col]:,}"
                else:
                    label = f"{normalized[row, col]:.2f}"
                text_color = "white" if values[row, col] > values.max() / 2 else "black"
                ax.text(col, row, label, ha="center", va="center", color=text_color, fontsize=7)
        ax.set_xticks(CLASS_IDS, labels=CLASS_NAMES, rotation=45, ha="right")
        ax.set_yticks(CLASS_IDS, labels=CLASS_NAMES)
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True")
        ax.set_title(title)
        fig.colorbar(image, ax=ax, shrink=0.7)
    fig.suptitle(f"{candidate_id} — {split}", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(directory / "confusion_matrix.png", dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    print(f"\n{candidate_id} | {split.upper()}")
    print(f"Macro-F1: {metrics['macro_f1']:.6f} | Accuracy: {metrics['accuracy']:.6f}")
    print(f"Prediction time: {predict_seconds:,.2f} s | Fit cost: {fit_seconds:,.2f} s")
    display(per_class_table.round(5))
    return metrics


## 3. Create Out-of-Fold Probabilities

Mỗi dòng train được dự đoán đúng một lần từ model chưa học dòng đó.
OOF macro-F1 chỉ để khảo sát, vì các nhãn OOF sẽ được dùng để fit meta-model; không coi điểm train của meta-model là đánh giá độc lập.
Kiểm tra giao nhau trên đầu vào thực sau median/bỏ cột hằng. Nếu phát sinh trùng mới, dừng thay vì tự xóa hàng.


In [ ]:
print("=" * 70)
print("GENERATE 5-FOLD OOF PROBABILITIES")
print("=" * 70)

OOF_DIR = OUTPUT_DIR / "oof"
OOF_DIR.mkdir()
oof_probabilities = {}
for name in BASE_NAMES:
    matrix = np.lib.format.open_memmap(
        OOF_DIR / f"{name}_probabilities.npy", mode="w+", dtype=np.float32,
        shape=(len(y_train), len(CLASS_NAMES)),
    )
    matrix[:] = np.nan
    oof_probabilities[name] = matrix
coverage = np.zeros((len(y_train), len(BASE_NAMES)), dtype=np.uint8)
fit_records = []
preprocessing_records = []
overlap_records = []

for fold_id in range(N_FOLDS):
    print(f"\n{'=' * 70}\nFOLD {fold_id + 1} / {N_FOLDS}\n{'=' * 70}")
    fit_indices = np.flatnonzero(fold_ids != fold_id)
    oof_indices = np.flatnonzero(fold_ids == fold_id)
    if len(np.intersect1d(train_groups[fit_indices], train_groups[oof_indices])) > 0:
        raise ValueError(f"Fold {fold_id}: fit và OOF có nhóm chung.")
    check_classes(y_train[fit_indices], f"fold {fold_id} fit")
    check_classes(y_train[oof_indices], f"fold {fold_id} OOF")

    prep_started = time.perf_counter()
    preprocessor, fit_values, prep_record = fit_preprocessor(
        X_train[fit_indices], f"fold_{fold_id}_fit"
    )
    hold_values = preprocessor.transform(X_train[oof_indices]).astype(np.float32, copy=False)
    if not np.isfinite(hold_values).all():
        raise ValueError(f"Fold {fold_id}: OOF có NaN/Inf sau tiền xử lý.")
    prep_seconds = time.perf_counter() - prep_started
    preprocessing_records.append(prep_record)
    overlap_records.append(check_effective_overlap(fit_values, hold_values, f"fold_{fold_id}_fit__oof"))
    print(f"Fit rows: {len(fit_indices):,} | OOF rows: {len(oof_indices):,}")
    print(f"Features kept: {fit_values.shape[1]} | Preprocessing: {prep_seconds:.2f} s")

    for model_index, name in enumerate(BASE_NAMES):
        model = clone(base_templates[name])
        started = time.perf_counter()
        with threadpool_limits(limits=N_JOBS):
            model.fit(fit_values, y_train[fit_indices])
        fit_seconds = time.perf_counter() - started
        started = time.perf_counter()
        probabilities = model_probabilities(model, hold_values)
        predict_seconds = time.perf_counter() - started
        if np.any(coverage[oof_indices, model_index] != 0):
            raise RuntimeError("Ghi đè dự đoán OOF.")
        oof_probabilities[name][oof_indices] = probabilities
        coverage[oof_indices, model_index] += 1
        oof_probabilities[name].flush()
        fit_records.append({
            "stage": "oof", "fold": fold_id, "model": name,
            "fit_seconds": fit_seconds, "predict_seconds": predict_seconds,
            "preprocess_seconds_shared": prep_seconds,
        })
        print(f"  {name:<4}: fit={fit_seconds:,.2f} s | OOF predict={predict_seconds:,.2f} s")
        del model, probabilities
        gc.collect()

    del preprocessor, fit_values, hold_values, fit_indices, oof_indices
    gc.collect()
    pd.DataFrame(fit_records).to_csv(OUTPUT_DIR / "training_times.csv", index=False)
    write_json(OUTPUT_DIR / "preprocessing_audit.json", preprocessing_records)
    write_json(OUTPUT_DIR / "effective_overlap_audit.json", overlap_records)

if not np.all(coverage == 1):
    raise RuntimeError("Mỗi dòng train phải có đúng 1 OOF prediction cho mỗi base model.")
for name in BASE_NAMES:
    if not np.isfinite(oof_probabilities[name]).all():
        raise RuntimeError(f"{name}: thiếu/xấu xác suất OOF.")
    oof_predicted = oof_probabilities[name].argmax(axis=1)
    score = f1_score(y_train, oof_predicted, labels=CLASS_IDS, average="macro", zero_division=0)
    print(f"{name} OOF macro-F1 (diagnostic): {score:.6f}")
np.save(OOF_DIR / "coverage.npy", coverage, allow_pickle=False)
np.save(OOF_DIR / "fold_ids.npy", fold_ids, allow_pickle=False)
del coverage, oof_predicted
gc.collect()
print("[OK] OOF coverage is exactly one for every row and base model.")


## 4. Refit Base Models on the Entire Training Set

Preprocessor cuối cùng học từ toàn train. Không dùng validation để early stopping hoặc fit model.
HGB dùng số vòng lặp đã chốt (`early_stopping=False`). Đây là các base models dùng khi suy luận.


In [ ]:
print("=" * 70)
print("REFIT BASE MODELS ON FULL TRAIN")
print("=" * 70)

started = time.perf_counter()
full_preprocessor, full_fit_values, prep_record = fit_preprocessor(X_train, "full_train")
val_values = full_preprocessor.transform(X_val).astype(np.float32, copy=False)
if not np.isfinite(val_values).all():
    raise ValueError("Validation có NaN/Inf sau tiền xử lý.")
full_preprocess_seconds = time.perf_counter() - started
preprocessing_records.append(prep_record)
overlap_records.append(check_effective_overlap(full_fit_values, val_values, "full_train__validation"))
joblib.dump(full_preprocessor, OUTPUT_DIR / "full_preprocessor.joblib")

# Lưu hash đầu vào thực để kiểm tra test sau khi freeze mà không giữ ma trận fit lớn.
np.save(OUTPUT_DIR / "full_train_input_hashes.npy", np.unique(feature_hashes(full_fit_values)), allow_pickle=False)
np.save(OUTPUT_DIR / "val_input_hashes.npy", np.unique(feature_hashes(val_values)), allow_pickle=False)

full_models = {}
val_probabilities = {}
base_fit_costs = {}
full_fit_times = {}
validation_records = []
BASE_DIR = OUTPUT_DIR / "base_models"
BASE_DIR.mkdir()

for name in BASE_NAMES:
    model = clone(base_templates[name])
    started = time.perf_counter()
    with threadpool_limits(limits=N_JOBS):
        model.fit(full_fit_values, y_train)
    full_fit_seconds = time.perf_counter() - started
    full_fit_times[name] = full_fit_seconds
    full_models[name] = model
    joblib.dump(model, BASE_DIR / f"{name}.joblib")

    started = time.perf_counter()
    probabilities = model_probabilities(model, val_values)
    base_predict_seconds = time.perf_counter() - started
    val_probabilities[name] = probabilities
    np.save(OUTPUT_DIR / f"validation_{name}_probabilities.npy", probabilities, allow_pickle=False)
    fit_records.append({
        "stage": "full_train", "fold": -1, "model": name,
        "fit_seconds": full_fit_seconds, "predict_seconds": base_predict_seconds,
        "preprocess_seconds_shared": full_preprocess_seconds,
    })
    base_fit_costs[name] = sum(row["fit_seconds"] for row in fit_records if row["model"] == name)
    # Validation timing cho base model gồm cả transform đầu vào, đo trong phần benchmark sau.
    metrics = evaluate_predictions(
        name, "validation", y_val, probabilities, base_predict_seconds, base_fit_costs[name]
    )
    metrics["kind"] = "base"
    validation_records.append(metrics)
    print(f"  Full-train fit: {full_fit_seconds:,.2f} seconds")

pd.DataFrame(fit_records).to_csv(OUTPUT_DIR / "training_times.csv", index=False)
write_json(OUTPUT_DIR / "preprocessing_audit.json", preprocessing_records)
write_json(OUTPUT_DIR / "effective_overlap_audit.json", overlap_records)
write_json(OUTPUT_DIR / "full_train_fit_times.json", full_fit_times)
del full_fit_values, val_values, model, probabilities
# Giữ full_models và full_preprocessor để xuất bundle cuối, không fit lại sau khi chọn meta.
gc.collect()
print("[OK] Full-train base models saved.")


## 5. Train Meta-Models on OOF and Select Using Validation

Lưới đã chốt: hai tổ hợp base models × ba cấu hình Logistic Regression.
Không train meta-model bằng dự đoán in-sample của full-train base models.
Bảng validation bao gồm base models để kiểm tra stacking có cải thiện thật hay không.
Thời gian validation ở bước này là thời gian các bước tương ứng; benchmark toàn pipeline được đo riêng sau freeze.


In [ ]:
print("=" * 70)
print("TRAIN META-MODELS FROM OOF PROBABILITIES")
print("=" * 70)

META_DIR = OUTPUT_DIR / "meta_models"
META_DIR.mkdir()
candidate_details = {}
for names in BASE_SUBSETS:
    oof_meta_input = meta_features(oof_probabilities, names)
    val_meta_input = meta_features(val_probabilities, names)
    for meta_config in META_CONFIGS:
        candidate_id = "STACK_" + "_".join(names) + "_" + meta_config["name"]
        meta_model = LogisticRegression(
            C=meta_config["C"], class_weight=meta_config["class_weight"],
            solver="lbfgs", max_iter=META_MAX_ITER, random_state=RANDOM_STATE,
        )
        started = time.perf_counter()
        with warnings.catch_warnings(record=True) as caught_warnings:
            warnings.simplefilter("always", ConvergenceWarning)
            with threadpool_limits(limits=N_JOBS):
                meta_model.fit(oof_meta_input, y_train)
        meta_fit_seconds = time.perf_counter() - started
        for warning in caught_warnings:
            if issubclass(warning.category, ConvergenceWarning):
                raise RuntimeError(
                    f"{candidate_id}: meta-model chưa hội tụ. "
                    "Tăng ngân sách trước một run mới; chưa freeze hoặc đọc test."
                )

        started = time.perf_counter()
        probabilities = model_probabilities(meta_model, val_meta_input)
        meta_predict_seconds = time.perf_counter() - started
        joblib.dump(meta_model, META_DIR / f"{candidate_id}.joblib")

        model_fit_cost = meta_fit_seconds
        prediction_cost = meta_predict_seconds
        for name in names:
            model_fit_cost += base_fit_costs[name]
            for row in fit_records:
                if row["stage"] == "full_train" and row["model"] == name:
                    prediction_cost += row["predict_seconds"]
        metrics = evaluate_predictions(
            candidate_id, "validation", y_val, probabilities, prediction_cost, model_fit_cost
        )
        metrics["kind"] = "stacking"
        validation_records.append(metrics)
        candidate_details[candidate_id] = {
            "base_names": names, "meta_config": meta_config,
            "meta_fit_seconds": meta_fit_seconds,
            "meta_file": f"meta_models/{candidate_id}.joblib",
        }
        del meta_model, probabilities
        gc.collect()
    del oof_meta_input, val_meta_input
    gc.collect()

validation_table = pd.DataFrame(validation_records)
validation_table = validation_table.sort_values(
    ["macro_f1", "candidate_id"], ascending=[False, True], kind="stable"
).reset_index(drop=True)
validation_table.to_csv(OUTPUT_DIR / "validation_comparison.csv", index=False)
write_json(OUTPUT_DIR / "candidate_details.json", candidate_details)

print("\nVALIDATION COMPARISON:")
display(validation_table.round(6))
print("Fit cost includes OOF and full-train model fits; shared preprocessing is reported separately.")
print("Inference timing here excludes shared preprocessing; end-to-end timing follows after freeze.")


## 6. Freeze the Best Stack and the Best Base Model

Chọn hai ứng viên bằng validation trước khi đọc giá trị/nhãn test.
Lưu model cùng preprocessing, mapping và thứ tự base/class để nạp lại mà không cần lớp Python tùy biến.
Nếu base model tốt hơn stacking trên validation, ghi rõ và chọn base làm ứng viên chính.


In [ ]:
print("=" * 70)
print("FREEZE SELECTED CANDIDATES BEFORE TEST")
print("=" * 70)

best_stack_row = validation_table.loc[validation_table["kind"] == "stacking"].iloc[0]
best_base_row = validation_table.loc[validation_table["kind"] == "base"].iloc[0]
best_stack_id = best_stack_row["candidate_id"]
best_base_id = best_base_row["candidate_id"]
primary_candidate_id = validation_table.iloc[0]["candidate_id"]

BUNDLE_DIR = OUTPUT_DIR / "frozen_models"
BUNDLE_DIR.mkdir()
frozen_candidates = []
for candidate_id, kind in [(best_stack_id, "stacking"), (best_base_id, "base")]:
    if kind == "stacking":
        names = candidate_details[candidate_id]["base_names"]
        meta_model = joblib.load(OUTPUT_DIR / candidate_details[candidate_id]["meta_file"])
    else:
        names = [candidate_id]
        meta_model = None
    selected_models = {}
    for name in names:
        selected_models[name] = full_models[name]

    bundle = {
        "protocol": "stacking_fixed_split_v1", "candidate_id": candidate_id, "kind": kind,
        "feature_names": feature_names, "class_names": CLASS_NAMES,
        "input_dtype": "float32", "base_names": names,
        "preprocessor": full_preprocessor, "base_models": selected_models,
        "meta_model": meta_model, "versions": versions,
        "data_run_id": data_config["run_id"], "threads": N_JOBS,
    }
    bundle_path = BUNDLE_DIR / f"{candidate_id}.joblib"
    # Golden reference lấy từ các model đang ở RAM trước khi nạp joblib.
    reference_values = full_preprocessor.transform(X_val[:GOLDEN_SAMPLE_SIZE]).astype(np.float32, copy=False)
    reference_blocks = []
    for name in names:
        reference_blocks.append(model_probabilities(selected_models[name], reference_values))
    if kind == "stacking":
        reference_probabilities = model_probabilities(meta_model, np.concatenate(reference_blocks, axis=1))
    else:
        reference_probabilities = reference_blocks[0]
    reference_path = BUNDLE_DIR / f"{candidate_id}_golden_reference.npy"
    np.save(reference_path, reference_probabilities, allow_pickle=False)
    joblib.dump(bundle, bundle_path)
    matching_row = validation_table.loc[validation_table["candidate_id"] == candidate_id].iloc[0]
    frozen_candidates.append({
        "candidate_id": candidate_id, "kind": kind,
        "bundle_file": str(bundle_path.relative_to(OUTPUT_DIR)),
        "sha256": sha256_file(bundle_path),
        "golden_reference_file": str(reference_path.relative_to(OUTPUT_DIR)),
        "golden_reference_sha256": sha256_file(reference_path),
        "validation_macro_f1": float(matching_row["macro_f1"]),
        "model_fit_seconds": float(matching_row["fit_seconds"]),
    })

freeze_record = {
    "run_id": RUN_ID, "data_run_id": data_config["run_id"],
    "frozen_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "selection": protocol["selection"], "primary_candidate_id": primary_candidate_id,
    "candidates": frozen_candidates,
    "protocol_sha256": sha256_file(OUTPUT_DIR / "protocol.json"),
    "validation_table_sha256": sha256_file(OUTPUT_DIR / "validation_comparison.csv"),
    "previous_validation_test_viewed": True,
}
write_json(OUTPUT_DIR / "frozen_candidates.json", freeze_record)
print(f"Best stack : {best_stack_id}")
print(f"Best base  : {best_base_id}")
print(f"Primary    : {primary_candidate_id}")
print("[OK] Selection and model checksums frozen. Test has not been evaluated.")
del bundle, selected_models, meta_model
# Các biến tạm chỉ dùng để tạo golden reference.
del reference_values, reference_blocks, reference_probabilities
# Nạp lại bundle đã freeze để kiểm tra tái sử dụng; không giữ thêm tất cả model trong RAM.
full_models.clear()
val_probabilities.clear()
oof_probabilities.clear()
del full_preprocessor
del train_groups, val_groups, test_groups
gc.collect()


## 7. Export CSV Prediction Code and Verify Reloaded Models

Xuất `predict_csv.py` cùng bundle. Kiểm tra joblib reload, CSV round-trip và mẫu golden trên validation trước khi mở test.
Gói suy luận không fit lại tiền xử lý hoặc model. CSV chỉ chứa các cột feature, có thể kèm cột `label` không dùng khi dự đoán.


In [ ]:
inference_source = r'''
import argparse
import csv
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from threadpoolctl import threadpool_limits


def load_bundle(path):
    bundle = joblib.load(path)
    current_versions = {
        "numpy": np.__version__, "pandas": pd.__version__,
        "sklearn": sklearn.__version__, "joblib": joblib.__version__,
    }
    for package, version in current_versions.items():
        if bundle["versions"][package] != version:
            raise RuntimeError(f"Use recorded {package} version: {bundle['versions'][package]}")
    return bundle


def read_csv_features(path, feature_names):
    with Path(path).open(newline="", encoding="utf-8-sig") as file:
        header = next(csv.reader(file))
    if len(header) != len(set(header)):
        raise ValueError("CSV has duplicate column names.")
    data = pd.read_csv(path)
    missing = sorted(set(feature_names) - set(data.columns))
    extra = sorted(set(data.columns) - set(feature_names) - {"label"})
    if missing or extra or len(data.columns) != len(set(data.columns)):
        raise ValueError(f"CSV schema mismatch: missing={missing}, extra={extra}")
    features = data.loc[:, feature_names].astype(np.float64)
    features = features.replace([np.inf, -np.inf], np.nan)
    with np.errstate(over="ignore", invalid="ignore"):
        values = features.to_numpy(dtype=np.float32)
    if np.isinf(values).any():
        raise ValueError("Float32 conversion overflowed.")
    return values


def predict_bundle(bundle, values, batch_size=25000):
    if values.ndim != 2 or values.shape[1] != len(bundle["feature_names"]):
        raise ValueError("Unexpected input feature count/order.")
    expected_classes = np.arange(len(bundle["class_names"]))
    result = np.empty((len(values), len(expected_classes)), dtype=np.float32)
    with threadpool_limits(limits=bundle["threads"]):
        for start in range(0, len(values), batch_size):
            stop = min(start + batch_size, len(values))
            with np.errstate(over="ignore", invalid="ignore"):
                batch = np.asarray(values[start:stop], dtype=np.float32)
            if np.isinf(batch).any():
                raise ValueError("Input has Inf or float32 overflow.")
            transformed = bundle["preprocessor"].transform(batch).astype(np.float32, copy=False)
            if not np.isfinite(transformed).all():
                raise ValueError("Preprocessing produced NaN/Inf.")
            blocks = []
            for name in bundle["base_names"]:
                model = bundle["base_models"][name]
                if not np.array_equal(model.classes_, expected_classes):
                    raise ValueError("Unexpected base class order.")
                probabilities = model.predict_proba(transformed).astype(np.float32)
                blocks.append(probabilities)
            if bundle["kind"] == "base":
                probabilities = blocks[0]
            else:
                meta_input = np.concatenate(blocks, axis=1)
                model = bundle["meta_model"]
                if not np.array_equal(model.classes_, expected_classes):
                    raise ValueError("Unexpected meta class order.")
                probabilities = model.predict_proba(meta_input).astype(np.float32)
            if not np.isfinite(probabilities).all():
                raise ValueError("Non-finite probabilities.")
            if np.any(probabilities < 0) or np.any(probabilities > 1):
                raise ValueError("Probabilities outside [0, 1].")
            if not np.allclose(probabilities.sum(axis=1), 1.0, atol=1e-5):
                raise ValueError("Probabilities do not sum to one.")
            result[start:stop] = probabilities
    return result


def predict_csv(bundle_path, input_path, output_path):
    bundle = load_bundle(bundle_path)
    values = read_csv_features(input_path, bundle["feature_names"])
    probabilities = predict_bundle(bundle, values)
    predicted = probabilities.argmax(axis=1)
    output = pd.DataFrame({"class_id": predicted})
    output["class_name"] = np.asarray(bundle["class_names"])[predicted]
    for class_id, class_name in enumerate(bundle["class_names"]):
        output[f"prob_{class_name}"] = probabilities[:, class_id]
    output.to_csv(output_path, index=False)
    return output


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--bundle", required=True, type=Path)
    parser.add_argument("--input", required=True, type=Path)
    parser.add_argument("--output", required=True, type=Path)
    args = parser.parse_args()
    predict_csv(args.bundle, args.input, args.output)
'''

predict_script = OUTPUT_DIR / "predict_csv.py"
predict_script.write_text(inference_source, encoding="utf-8")
spec = importlib.util.spec_from_file_location("stacking_csv_prediction", predict_script)
prediction_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(prediction_module)
print(f"Prediction code saved to: {predict_script}")


In [ ]:
print("=" * 70)
print("RELOAD AND CSV GOLDEN SAMPLE CHECKS — VALIDATION ONLY")
print("=" * 70)

GOLDEN_DIR = OUTPUT_DIR / "golden_samples"
GOLDEN_DIR.mkdir()
golden_indices = np.arange(min(GOLDEN_SAMPLE_SIZE, len(y_val)))
golden_values = np.array(X_val[golden_indices], copy=True)
golden_csv = GOLDEN_DIR / "input.csv"
pd.DataFrame(golden_values, columns=feature_names).to_csv(golden_csv, index=False, float_format="%.17g")
golden_from_csv = prediction_module.read_csv_features(golden_csv, feature_names)
np.testing.assert_allclose(golden_values, golden_from_csv, rtol=0, atol=0, equal_nan=True)

reload_records = []
for selected in frozen_candidates:
    bundle_path = OUTPUT_DIR / selected["bundle_file"]
    if sha256_file(bundle_path) != selected["sha256"]:
        raise RuntimeError("Bundle thay đổi sau freeze.")
    reference_path = OUTPUT_DIR / selected["golden_reference_file"]
    if sha256_file(reference_path) != selected["golden_reference_sha256"]:
        raise RuntimeError("Golden reference thay đổi sau freeze.")
    before = np.load(reference_path, allow_pickle=False)
    reloaded_bundle = prediction_module.load_bundle(bundle_path)
    after = prediction_module.predict_bundle(reloaded_bundle, golden_from_csv, PREDICT_BATCH_SIZE)
    np.testing.assert_allclose(before, after, rtol=1e-6, atol=1e-7)
    np.testing.assert_array_equal(before.argmax(axis=1), after.argmax(axis=1))

    output_path = GOLDEN_DIR / f"{selected['candidate_id']}_predictions.csv"
    csv_output = prediction_module.predict_csv(bundle_path, golden_csv, output_path)
    csv_probability_columns = []
    for class_name in CLASS_NAMES:
        csv_probability_columns.append(f"prob_{class_name}")
    np.testing.assert_allclose(csv_output[csv_probability_columns].to_numpy(), after, rtol=1e-6, atol=1e-7)
    reload_records.append({"candidate_id": selected["candidate_id"], "rows": len(golden_values), "passed": True})
    print(f"[OK] {selected['candidate_id']}: joblib reload and CSV golden sample")
    del reloaded_bundle, before, after, csv_output
    gc.collect()
write_json(OUTPUT_DIR / "reload_checks.json", reload_records)


## 8. End-to-End Validation Benchmark After Freeze

Đo từ feature chưa điền thiếu qua preprocessor → base models → meta-model, bằng bundle đã nạp lại.
Thời gian gồm transform/predict, không gồm load model, ghi báo cáo hoặc vẽ biểu đồ.
Kiểm tra dự đoán của bundle khớp kết quả validation đã dùng để chọn; không chọn lại ứng viên ở bước này.


In [ ]:
end_to_end_validation = []
for selected in frozen_candidates:
    bundle = prediction_module.load_bundle(OUTPUT_DIR / selected["bundle_file"])
    started = time.perf_counter()
    probabilities = prediction_module.predict_bundle(bundle, X_val, PREDICT_BATCH_SIZE)
    elapsed = time.perf_counter() - started
    score = float(f1_score(y_val, probabilities.argmax(axis=1), labels=CLASS_IDS, average="macro", zero_division=0))
    if not np.isclose(score, selected["validation_macro_f1"], rtol=0, atol=1e-12):
        raise RuntimeError("Reloaded bundle không khớp validation macro-F1 trước freeze.")
    end_to_end_validation.append({
        "candidate_id": selected["candidate_id"], "rows": len(y_val),
        "predict_seconds": elapsed, "predict_ms_per_row": elapsed / len(y_val) * 1000,
        "macro_f1": score,
    })
    del bundle, probabilities
    gc.collect()
pd.DataFrame(end_to_end_validation).to_csv(OUTPUT_DIR / "validation_end_to_end_timing.csv", index=False)
print("END-TO-END VALIDATION TIMING:")
display(pd.DataFrame(end_to_end_validation).round(6))


## 9. Final Test Evaluation — Frozen Candidates Only

Đây là lần đầu notebook đọc X_test/y_test. Trước tiên kiểm tra freeze/checksum và giao nhau đầu vào thực.
Chỉ đánh giá best stacking và best base đã chốt. Báo cáo cả hai; ứng viên chính vẫn do validation quyết định.
Không refit trên train+validation, không đổi cấu hình theo test. Test hiện tại có giới hạn do lịch sử đã từng được xem.


In [ ]:
print("=" * 70)
print("FINAL TEST — CANDIDATES ALREADY FROZEN")
print("=" * 70)

frozen = json.loads((OUTPUT_DIR / "frozen_candidates.json").read_text())
if sha256_file(OUTPUT_DIR / "protocol.json") != frozen["protocol_sha256"]:
    raise RuntimeError("Protocol thay đổi sau freeze.")
if sha256_file(OUTPUT_DIR / "validation_comparison.csv") != frozen["validation_table_sha256"]:
    raise RuntimeError("Bảng validation thay đổi sau freeze.")
if (OUTPUT_DIR / "test_evaluation_started.json").exists():
    raise RuntimeError("Run này đã bắt đầu đánh giá test. Không chạy lại để chọn cấu hình.")

X_test = np.load(DATA_DIR / "X_test_unprocessed.npy", mmap_mode="r", allow_pickle=False)
y_test = np.load(DATA_DIR / "y_test.npy", allow_pickle=False)
if X_test.dtype != np.float32 or X_test.shape != (len(y_test), len(feature_names)):
    raise ValueError("Test shape/dtype không khớp.")
if list(X_test.shape) != data_config["artifacts"]["test"]["shape"]:
    raise ValueError("Test shape khác config.")
check_classes(y_test, "test")

# Hai bundle dùng chung full-train preprocessor đã freeze.
first_selected = frozen["candidates"][0]
first_bundle_path = OUTPUT_DIR / first_selected["bundle_file"]
if sha256_file(first_bundle_path) != first_selected["sha256"]:
    raise RuntimeError("Bundle thay đổi sau freeze.")
bundle = prediction_module.load_bundle(first_bundle_path)
test_hashes = np.empty(len(X_test), dtype="V16")
for start in range(0, len(X_test), PREDICT_BATCH_SIZE):
    stop = min(start + PREDICT_BATCH_SIZE, len(X_test))
    batch = np.asarray(X_test[start:stop], dtype=np.float32)
    if np.isinf(batch).any():
        raise ValueError("Test input chứa Inf.")
    transformed = bundle["preprocessor"].transform(batch).astype(np.float32, copy=False)
    if not np.isfinite(transformed).all():
        raise ValueError("Test có NaN/Inf sau tiền xử lý.")
    test_hashes[start:stop] = feature_hashes(transformed)
unique_test_hashes = np.unique(test_hashes)
for reference_file in ["full_train_input_hashes.npy", "val_input_hashes.npy"]:
    reference_hashes = np.load(OUTPUT_DIR / reference_file, mmap_mode="r", allow_pickle=False)
    overlap_count = len(np.intersect1d(reference_hashes, unique_test_hashes, assume_unique=True))
    if overlap_count > 0:
        raise ValueError(f"Test sau preprocessing trùng {overlap_count} nhóm với {reference_file}.")
    overlap_records.append({"description": f"test__{reference_file}", "overlap_groups": 0})
write_json(OUTPUT_DIR / "effective_overlap_audit.json", overlap_records)
del bundle, batch, transformed, test_hashes, unique_test_hashes, reference_hashes

# Khóa lần đánh giá trước khi tính metric; nếu cell lỗi thì chưa thể coi run complete.
write_json(OUTPUT_DIR / "test_evaluation_started.json", {
    "frozen_candidates_sha256": sha256_file(OUTPUT_DIR / "frozen_candidates.json"),
    "started_at": time.strftime("%Y-%m-%d %H:%M:%S"),
})
test_records = []
for selected in frozen["candidates"]:
    path = OUTPUT_DIR / selected["bundle_file"]
    if sha256_file(path) != selected["sha256"]:
        raise RuntimeError("Bundle thay đổi sau freeze.")
    bundle = prediction_module.load_bundle(path)
    started = time.perf_counter()
    probabilities = prediction_module.predict_bundle(bundle, X_test, PREDICT_BATCH_SIZE)
    elapsed = time.perf_counter() - started
    metrics = evaluate_predictions(
        selected["candidate_id"], "test", y_test, probabilities,
        elapsed, selected["model_fit_seconds"],
    )
    metrics["kind"] = selected["kind"]
    metrics["primary_selected_on_validation"] = selected["candidate_id"] == frozen["primary_candidate_id"]
    test_records.append(metrics)
    del bundle, probabilities
    gc.collect()
test_table = pd.DataFrame(test_records)
test_table.to_csv(OUTPUT_DIR / "test_comparison.csv", index=False)
print("\nFINAL TEST COMPARISON (SELECTION REMAINS BASED ON VALIDATION):")
display(test_table.round(6))


# Training Summary Report


In [ ]:
print("=" * 70)
print("PHASE 3: STACKING SUMMARY REPORT")
print("=" * 70)
print(f"Data run     : {data_config['run_id']}")
print(f"Training run : {RUN_ID}")
print(f"Primary model: {frozen['primary_candidate_id']} (selected on validation)")
print("\n[1] VALIDATION")
display(validation_table[["candidate_id", "kind", "macro_f1", "accuracy"]].round(6))
print("\n[2] TEST — FROZEN CANDIDATES")
display(test_table.round(6))

fit_time_table = pd.DataFrame(fit_records)
shared_preprocessing_seconds = fit_time_table.drop_duplicates(["stage", "fold"])["preprocess_seconds_shared"].sum()
print("\n[3] TRAINING COST")
print(f"Base model fits (OOF + full train): {fit_time_table['fit_seconds'].sum():,.2f} s")
print(f"Shared preprocessing             : {shared_preprocessing_seconds:,.2f} s")
print("Detailed model fit/predict times  : training_times.csv")
print("End-to-end validation inference   : validation_end_to_end_timing.csv")
print("Test inference includes preprocessing and all selected model stages.")

print("\n[4] REUSE AND CHECKS")
print("OOF coverage, class order, input checksums and overlap checks passed.")
print("Reload and validation CSV golden sample checks passed.")
print("Prediction helper: predict_csv.py")
print("Example: python predict_csv.py --bundle frozen_models/<id>.joblib --input data.csv --output predictions.csv")
print("Use versions recorded in protocol.json and the selected bundle.")
print("\n[5] LIMITATIONS")
print("Baseline validation/test were previously viewed; the existing Kaggle test is not a fresh independent test.")
print("Feature groups do not establish independence by device/session/time.")

# Chỉ hoàn tất khi test, reload và các báo cáo trên đều thành công.
write_json(OUTPUT_DIR / "run_status.json", {
    "status": "complete", "run_id": RUN_ID, "data_run_id": data_config["run_id"],
    "primary_candidate_id": frozen["primary_candidate_id"],
    "finished_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "total_seconds": round(time.perf_counter() - SESSION_START, 3),
})
print(f"\n[OK] STACKING RUN COMPLETED: {OUTPUT_DIR}")
